> **통합 안내:** 이 노트북의 블록 생성은 `02_블록만들기.ipynb`의 8절로 옮겼습니다. 신고기준 블록 번호가 바뀌었습니다: 신고범위 b01 · 진단기준 b02 · 신고시기 b03 · 신고방법 b04 (이 노트북의 이전 번호 b04 진단기준은 더 이상 쓰지 않습니다). 기록용으로 남겨 둡니다.

# P3(추가). 신고를 위한 진단기준 블록 — 홍역 · 성홍열

## 이 노트북이 하는 일
- 신고범위의 "환자, 의사환자"가 무엇인지 근거를 대기 위해, **진단기준 행**을 블록(b04)으로 만듭니다.
- 원본 대조: 홍역 PDF 89쪽(인쇄 81쪽), 성홍열 PDF 104쪽(인쇄 96쪽)의 이미지로 확인했습니다. 표 행의 정의가 이미지와 같습니다.
- 확인 방식: 변환본에는 같은 정의가 **두 번** 나옵니다. 표 행은 `○ 의사환자 / - 의심환자`, 쪽 요약 목록은 `- 의사환자 / - 의심환자`로 기호가 다릅니다. 그래서 **문장 조각 단위**로 원문에 있는지 확인합니다. 전체 문장 한 번 검사는 기호 차이 때문에 실패할 수 있습니다.
- 이 블록은 **환자·의사환자의 정의**만 근거로 씁니다. 신고 주체(누가 신고할지)의 근거로는 쓰지 않습니다.

In [1]:
import re
import json
from preprocess.prevention_legal.pages import DOCS, excerpt_in_page, read_page

KEY = "diagnosis"
DOC_ID = DOCS[KEY][0]
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
existing = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]
existing_ids = {b["chunk_id"] for b in existing}
print("기존 블록:", len(existing))

기존 블록: 14


## 1. 표 행 읽기
문서의 `| 신고를 위한 진단기준 |` 행을 쪽별로 찾아 셀 글을 꺼냅니다. `<br>`은 공백으로 바꿉니다.

In [2]:
raw = open("../../data/raw/prevention_legal/2026년 법정감염병 진단·신고기준 지침_최종.md", encoding="utf-8").read()
parts = re.split(r"^## PDF (\d+)페이지", raw, flags=re.M)
bodies = {int(parts[i]): parts[i + 1] for i in range(1, len(parts), 2)}

def definition_cell(page):
    # 해당 쪽의 '신고를 위한 진단기준' 표 행에서 셀 글만 꺼낸다
    line = next(l for l in bodies[page].splitlines() if l.startswith("| 신고를 위한 진단기준 |"))
    cell = line.split("|")[2]
    return " ".join(cell.replace("<br>", " ").split())

for page in (89, 104):
    print(page, "쪽:", definition_cell(page)[:70], "...")

89 쪽: ○ 환자 : 홍역에 부합되는 임상증상을 나타내면서 진단을 위한 검사기준에 따라 감염병병원체 감염이 확인된 사람 ○ 의사환자 : ...
104 쪽: ○ 환자 : 성홍열에 부합되는 임상증상을 나타내면서 확인 진단을 위한 검사기준에 따라 감염병병원체 감염이 확인된 사람 ○ 의사 ...


## 2. 조각별 원문 확인
`○`나 ` - ` 로 나눈 조각 하나하나가 해당 쪽 원문에 있는지 봅니다. 조각 하나라도 없으면 저장하지 않습니다.

In [3]:
PLAN = [(89, "81", "21. [제2급-3] 홍역(Measles) · 신고를 위한 진단기준"),
        (104, "96", "36. [제2급-18] 성홍열(Scarlet fever) · 신고를 위한 진단기준")]

new_blocks = []
for page, printed, section in PLAN:
    text = definition_cell(page)
    # 목록 기호(○)와 하위 목록 기호( - )로 조각을 나눈다
    pieces = [p.strip() for p in re.split(r"○|\s-\s", text) if p.strip()]
    for piece in pieces:
        ok = excerpt_in_page(piece, KEY, page)
        print(("원문 확인 " if ok else "확인 필요 "), f"PDF {page}쪽 | {piece[:40]}")
        assert ok, f"{page}쪽에서 조각을 찾지 못했습니다: {piece}"
    new_blocks.append({
        "chunk_id": f"{DOC_ID}:p{page:04d}:b04",
        "document_id": DOC_ID,
        "text": text,
        "pdf_page": page,
        "printed_page": int(printed),
        "section": section,
    })
print("새 블록:", [b["chunk_id"] for b in new_blocks])

원문 확인  PDF 89쪽 | 환자 : 홍역에 부합되는 임상증상을 나타내면서 진단을 위한 검사기준에 따
원문 확인  PDF 89쪽 | 의사환자 : 임상증상 및 역학적 연관성을 감안하여 홍역이 의심되나 진단을
원문 확인  PDF 104쪽 | 환자 : 성홍열에 부합되는 임상증상을 나타내면서 확인 진단을 위한 검사기
원문 확인  PDF 104쪽 | 의사환자
원문 확인  PDF 104쪽 | 의심환자 : 임상증상 및 역학적 연관성을 감안하여 성홍열이 의심되나 진단
원문 확인  PDF 104쪽 | 추정환자 : 임상증상 및 역학적 연관성을 감안하여 성홍열이 의심되며, 추
새 블록: ['kdca_diagnosis_reporting_2026:p0089:b04', 'kdca_diagnosis_reporting_2026:p0104:b04']


## 3. 저장 (기존 블록은 지우지 않고, 같은 chunk_id 는 추가하지 않는다)

In [4]:
to_add = [b for b in new_blocks if b["chunk_id"] not in existing_ids]
with open(BLOCKS, "a", encoding="utf-8") as f:
    for b in to_add:
        f.write(json.dumps(b, ensure_ascii=False) + chr(10))
print("추가 저장:", len(to_add), "개 | 전체 블록:", sum(1 for _ in open(BLOCKS, encoding="utf-8")))

추가 저장: 2 개 | 전체 블록: 16
